In [1]:
from sentence_transformers import SentenceTransformer, util
import os
import pandas as pd
import pickle
import time
import torch
from annoy import AnnoyIndex

In [2]:
data = pd.read_csv('train_data.csv')

In [3]:
model_name = 'paraphrase-albert-small-v2'
model = SentenceTransformer(model_name)

In [4]:
max_corpus_size = data.shape[0]
n_trees = 256           #Number of trees used for Annoy. More trees => better recall, worse run-time
embedding_size = 768    #Size of embeddings
top_k_hits = 10 

In [5]:
annoy_index_path = 'quora-embeddings-{}-size-{}-annoy_index-trees-{}.ann'.format(model_name.replace('/', '_'), max_corpus_size,n_trees)
embedding_cache_path = 'quora-embeddings-{}-size-{}.pkl'.format(model_name.replace('/', '_'), max_corpus_size)

In [6]:
if not os.path.exists(embedding_cache_path):
    paragraphs = list(data['Paragraph'].unique())
    embeddings = model.encode(paragraphs,show_progress_bar=True,convert_to_numpy=True,device='cuda')
    with open(embedding_cache_path,'wb') as fout:
        pickle.dump({'paragraphs':paragraphs,'embeddings':embeddings},fout)
else:
    with open(embedding_cache_path,'rb') as fin:
        file = pickle.load(fin)
        embeddings = file['embeddings']
        paragraphs = file['paragraphs']

Batches:   0%|          | 0/487 [00:00<?, ?it/s]

In [8]:
if not os.path.exists(annoy_index_path):
    # Create Annoy Index
    print("Create Annoy index with {} trees. This can take some time.".format(n_trees))
    annoy_index = AnnoyIndex(embedding_size, 'angular')

    for i in range(len(embeddings)):
        annoy_index.add_item(i, embeddings[i])

    annoy_index.build(n_trees)
    annoy_index.save(annoy_index_path)
else:
    #Load Annoy Index from disc
    annoy_index = AnnoyIndex(embedding_size, 'angular')
    annoy_index.load(annoy_index_path)

Create Annoy index with 256 trees. This can take some time.


In [10]:
corpus_embeddings = torch.from_numpy(embeddings)

In [18]:
ques = data['Question'][800]
start_time = time.time()
question_embedding = model.encode(ques)

corpus_ids, scores = annoy_index.get_nns_by_vector(question_embedding, top_k_hits, include_distances=True)
hits = []
for id, score in zip(corpus_ids, scores):
    hits.append({'corpus_id': id, 'score': 1-((score**2) / 2)})
end_time = time.time()

In [28]:
print("Input question:", ques,'\n')
print("Results (after {:.3f} seconds):".format(end_time-start_time),'\n')
for hit in hits[0:3]:
    print("Score :{:.3f}\n\n{}".format(hit['score'], paragraphs[hit['corpus_id']]))
    print('---------------------------------------------------------------\n\n')

Input question: Modern medicine indicates Chopin may have suffered from what condition? 

Results (after 0.054 seconds): 

Score :0.686

From 1842 onwards, Chopin showed signs of serious illness. After a solo recital in Paris on 21 February 1842, he wrote to Grzymała: "I have to lie in bed all day long, my mouth and tonsils are aching so much." He was forced by illness to decline a written invitation from Alkan to participate in a repeat performance of the Beethoven Seventh Symphony arrangement at Erard's on 1 March 1843. Late in 1844, Charles Hallé visited Chopin and found him "hardly able to move, bent like a half-opened penknife and evidently in great pain", although his spirits returned when he started to play the piano for his visitor. Chopin's health continued to deteriorate, particularly from this time onwards. Modern research suggests that apart from any other illnesses, he may also have suffered from temporal lobe epilepsy.
-----------------------------------------------------

In [29]:
# Approximate Nearest Neighbor (ANN) is not exact, it might miss entries with high cosine similarity
# Here, we compute the recall of ANN compared to the exact results
correct_hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k_hits)[0]
correct_hits_ids = set([hit['corpus_id'] for hit in correct_hits])

In [32]:
#Compute recall
ann_corpus_ids = set(corpus_ids)
if len(ann_corpus_ids) != len(correct_hits_ids):
    print("Approximate Nearest Neighbor returned a different number of results than expected")

recall = len(ann_corpus_ids.intersection(correct_hits_ids)) / len(correct_hits_ids)
print("\nApproximate Nearest Neighbor Recall@{}: {:.2f}".format(top_k_hits, recall * 100))

if recall < 1:
    print("Missing results:")
    for hit in correct_hits[0:top_k_hits]:
        if hit['corpus_id'] not in ann_corpus_ids:
            print("\t{:.3f}\t{}".format(hit['score'], corpus_sentences[hit['corpus_id']]))
print("\n\n========\n")


Approximate Nearest Neighbor Recall@10: 100.00





In [59]:
index_dict = {}
for i,para in enumerate(data['Paragraph'].unique()):
    index_dict[para] = i
data['para_index'] = data['Paragraph'].map(index_dict)

results = pd.DataFrame()
questions = data['Question']
original_paras = data['Paragraph']
answer_avail = data['Answer_possible'].apply(lambda x: int(x))

predicted_paras = []
scores_actual_para = []
scores_top_pred = []
incorrect_clf = []
top = []
question_embeddings = model.encode(data['Question'].values.tolist(),device='cuda',show_progress_bar=True,convert_to_numpy=True)

for i in range(data.shape[0]):
    
    question_embedding = question_embeddings[i]
    corpus_ids, scores = annoy_index.get_nns_by_vector(question_embedding, top_k_hits, include_distances=True)
    scores_top_pred.append(1-((scores[0]**2) / 2))
    
    if corpus_ids[0] == data['para_index'][i]:
        incorrect_clf.append(0)
        predicted_paras.append(data['Paragraph'][i])
        scores_actual_para.append(1-((scores[0]**2) / 2))
        top.append(1)
    elif data['para_index'][i] not in corpus_ids:
        incorrect_clf.append(1)
        predicted_paras.append('')
        scores_actual_para.append(0)
        top.append(-1)
        
    else:
        incorrect_clf.append(1)
        predicted_paras.append(data[data['para_index']==corpus_ids[0]].iloc[0,2])
        top.append(corpus_ids.index(data['para_index'][i])+1)
        scores_actual_para.append(1-((scores[corpus_ids.index(data['para_index'][i])]**2) / 2))
        
results['questions'] = questions
results['original_paras'] = original_paras
results['answer_avail'] = answer_avail
results['predicted_paras'] = predicted_paras
results['scores_actual_para'] = scores_actual_para
results['scores_top_pred'] = scores_top_pred
results['incorrect_clf'] = incorrect_clf
results['top'] = top

Batches:   0%|          | 0/2346 [00:00<?, ?it/s]

In [60]:
results.to_csv(f'results-{model_name}.csv')

In [2]:
results = pd.read_csv('results-paraphrase-albert-small-v2.csv')

In [3]:
results

,Unnamed: 0,questions,original_paras,answer_avail,predicted_paras,scores_actual_para,scores_top_pred,incorrect_clf,top
0,0,When did Beyonce leave Destiny's Child and bec...,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,1,Following the disbandment of Destiny's Child i...,0.620259,0.680186,1,6
1,1,What album made her a worldwide known artist?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,1,NaN,0.000000,0.601581,1,-1
2,2,Who managed the Destiny's Child group?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,1,NaN,0.000000,0.481828,1,-1
3,3,When did Beyoncé rise to fame?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,1,In The New Yorker music critic Jody Rosen desc...,0.581003,0.626787,1,10
4,4,What role did Beyoncé have in Destiny's Child?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,1,Following the disbandment of Destiny's Child i...,0.624931,0.702009,1,3
...,...,...,...,...,...,...,...,...,...
75050,75050,How many quarks and leptons are there?,These quarks and leptons interact through four...,0,The Standard Model groups matter particles int...,0.636361,0.640886,1,2
75051,75051,What model satisfactorily explains gravity?,These quarks and leptons interact through four...,0,Indian astronomer and mathematician Aryabhata ...,0.507802,0.572950,1,2
75052,75052,Mass and energy can always be compared to what?,These quarks and leptons interact through four...,0,NaN,0.000000,0.593187,1,-1
75053,75053,Physics has broadly agreed on the definition o...,"The term ""matter"" is used throughout physics i...",0,NaN,0.000000,0.517152,1,-1


In [4]:
results['incorrect_clf'].value_counts()

1    45835
0    29220
Name: incorrect_clf, dtype: int64

In [5]:
results['top'].value_counts()

 1     29220
-1     26581
 2      6691
 3      3416
 4      2390
 5      1769
 6      1369
 7      1065
 8       976
 9       813
 10      765
Name: top, dtype: int64